In [2]:
import sys, time, re, json, pathlib
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

DATA = pathlib.Path("data"); DATA.mkdir(exist_ok=True)
GENES = ["CYP2C19", "CYP2C9", "VKORC1", "DPYD", "TPMT"]
MV = "https://myvariant.info/v1"
AVI_FILE = DATA / "alphagenome_avi.tsv"

ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

FIELDS = ",".join([
    "dbsnp.rsid",                        # the rs number
    "snpeff.ann.effect",                 # consequence term
    "cadd.phred",                        # CADD
    "dbnsfp.revel.score",                # REVEL
    "dbnsfp.alphamissense",              # AlphaMissense
    "dbnsfp.aa.ref", "dbnsfp.aa.alt", "dbnsfp.aa.pos", "dbnsfp.genename",
    "gnomad_genome.af.af", "gnomad_exome.af.af",
    "clinvar.rcv.clinical_significance",
    "clinvar.rcv.review_status",
    "clinvar.gene.symbol",
    "clinvar.variant_id",                # join key for CADD (see Section 3)
])

def first(x):
    """Some fields come back as a single value, some as a list. Take the first."""
    return (x[0] if x else None) if isinstance(x, list) else x

def dig(d, *path):
    """Walk a nested dict safely. Returns None if any level is missing."""
    for k in path:
        if not isinstance(d, dict):
            return None
        d = d.get(k)
        if d is None:
            return None
    return d

# ClinVar review status, as the familiar star rating
STARS = {"practice guideline": 4,
         "reviewed by expert panel": 3,
         "criteria provided, multiple submitters, no conflicts": 2,
         "criteria provided, conflicting classifications": 1,
         "criteria provided, conflicting interpretations": 1,
         "criteria provided, single submitter": 1,
         "no assertion criteria provided": 0,
         "no classification provided": 0}

# snpEff reports one effect per transcript. Keep the most severe one.
SEVERITY = ["stop_gained", "frameshift_variant", "splice_acceptor_variant",
            "splice_donor_variant", "start_lost", "stop_lost", "missense_variant",
            "splice_region_variant", "synonymous_variant", "5_prime_UTR_variant",
            "3_prime_UTR_variant", "intron_variant", "upstream_gene_variant",
            "downstream_gene_variant", "intergenic_region"]

def all_effects(h):
    """Every effect snpEff reported, across every transcript, split into single terms."""
    ann = dig(h, "snpeff", "ann")
    raw = [a.get("effect") for a in (ann if isinstance(ann, list) else [ann]) if isinstance(a, dict)]
    terms = []
    for e in raw:                       # snpEff emits compound terms joined by "&"
        if e:
            terms += [t for t in e.split("&") if t != "structural_interaction_variant"]
    return sorted(set(terms))

def worst_effect(h):
    """Collapse to ONE term: the most severe across all transcripts."""
    terms = all_effects(h)
    if not terms:
        return None
    return min(terms, key=lambda t: SEVERITY.index(t) if t in SEVERITY else 99)

def to_row(h, gene=None):
    aaref, aaalt = first(dig(h, "dbnsfp", "aa", "ref")), first(dig(h, "dbnsfp", "aa", "alt"))
    aapos = first(dig(h, "dbnsfp", "aa", "pos"))
    rcv = first(dig(h, "clinvar", "rcv")) or {}   # one record per submission; take the first
    rev = rcv.get("review_status")
    am = dig(h, "dbnsfp", "alphamissense", "score")
    return {
        "hgvs_g":        h.get("_id"),
        "rsid":          dig(h, "dbsnp", "rsid"),
        "gene":          first(dig(h, "clinvar", "gene", "symbol")) or first(dig(h, "dbnsfp", "genename")) or gene,
        "clinvar_id":    dig(h, "clinvar", "variant_id"),
        "consequence":   worst_effect(h),
        "consequence_all": ";".join(all_effects(h)),   # keep the full picture
        "aa_change":     f"{aaref}{aapos}{aaalt}" if (aaref and aaalt and aapos) else None,
        "cadd_phred":    dig(h, "cadd", "phred"),
        "revel":         first(dig(h, "dbnsfp", "revel", "score")),
        "alphamissense": first(am) if isinstance(am, list) else am,
        "gnomad_af":     dig(h, "gnomad_genome", "af", "af") or dig(h, "gnomad_exome", "af", "af"),
        "clinvar_sig":   rcv.get("clinical_significance"),
        "clinvar_stars": STARS.get(str(rev).lower(), 0),
    }

FORCE_REFETCH = False

# Known pharmacogenomic alleles, so we can check the fetch worked and
# recognise them later.
ANCHORS = {"rs1799853": "CYP2C9*2", "rs1057910": "CYP2C9*3",
           "rs9923231": "VKORC1 -1639G>A", "rs4244285": "CYP2C19*2",
           "rs12248560": "CYP2C19*17", "rs3918290": "DPYD*2A",
           "rs67376798": "DPYD c.2846A>T", "rs1142345": "TPMT*3C"}

def fetch_gene(gene, verbose=True):
    """Search ClinVar for one gene, annotate every hit, return a tidy DataFrame."""
    r = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg38",
        "fields": "_id", "size": 1000})
    r.raise_for_status()
    ids = [h["_id"] for h in r.json().get("hits", [])]
    if verbose:
        print(f"  {gene:9s} {len(ids):>5} variants found", end="", flush=True)

    rows = []
    for i in range(0, len(ids), 500):
        rr = requests.post(f"{MV}/variant", timeout=180,
                           data={"ids": ",".join(ids[i:i+500]),
                                 "fields": FIELDS, "assembly": "hg38"})
        rr.raise_for_status()
        rows += [to_row(h, gene) for h in rr.json()]
        time.sleep(0.3)                        # be polite to a free API

    df = pd.DataFrame(rows).drop_duplicates("hgvs_g")

    # MyVariant.info only carries genome-wide CADD on hg19. Ask the same
    # question on hg19 and join on the ClinVar variant ID, which is build-free.
    r19 = requests.get(f"{MV}/query", timeout=90, params={
        "q": f"clinvar.gene.symbol:{gene}", "assembly": "hg19",
        "fields": "clinvar.variant_id,cadd.phred", "size": 1000})
    r19.raise_for_status()
    hits19 = r19.json().get("hits", [])
    cadd = {}
    for h in hits19:
        vid = first(dig(h, "clinvar", "variant_id"))      # occasionally a list
        ph = dig(h, "cadd", "phred")
        if vid is not None and ph is not None:
            cadd[vid] = ph
    df["cadd_phred"] = df.clinvar_id.map(cadd)

    if len(ids) >= 1000 or len(hits19) >= 1000:
        print(f"\n  [!] {gene} hit the 1000-result cap. The two builds may have been")
        print( "      truncated to different sets, so some CADD scores will be missing.")
    if verbose:
        print(f"  ->  {len(df)} annotated")
    return df

def numeric(df):
    """Force the score columns to numbers. Strings here break everything later."""
    for c in ["cadd_phred", "revel", "alphamissense", "spliceai_dg",
              "spliceai_dl", "avi", "gnomad_af", "clinvar_stars"]:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clinvar_label(sig):
    if not isinstance(sig, str): return "none"
    s = sig.lower()
    if "conflict" in s: return "conflicting"
    if "pathogenic" in s and "non-pathogenic" not in s: return "P/LP"
    if "benign" in s: return "B/LB"
    if "uncertain" in s: return "VUS"
    return "other"          # includes "drug response"

STRENGTH = [(18.7, "strong, pathogenic"), (4.3, "moderate, pathogenic"),
            (2.08, "supporting, pathogenic"), (0.48, "no evidence either way"),
            (0.23, "supporting, benign"), (0.053, "moderate, benign"),
            (0.0, "strong, benign")]

def strength_for(lr):
    if lr != lr: return "undefined"
    for cut, name in STRENGTH:
        if lr >= cut: return name
    return "strong, benign"

def calibrate(df, score_col, bands):
    d = df[df.label.isin(["P/LP", "B/LB"]) & df[score_col].notna()].copy()
    d["band"] = pd.cut(d[score_col], bands)
    n_p, n_b = (d.label == "P/LP").sum(), (d.label == "B/LB").sum()
    g = d.groupby("band", observed=False)
    out = pd.DataFrame({"n": g.size(),
                        "n_PLP": g.label.apply(lambda s: (s == "P/LP").sum()),
                        "n_BLB": g.label.apply(lambda s: (s == "B/LB").sum())})
    with np.errstate(divide="ignore", invalid="ignore"):
        out["LR+"] = ((out.n_PLP / max(n_p, 1)) / (out.n_BLB / max(n_b, 1))).round(2)
    out["evidence"] = out["LR+"].map(strength_for)
    out.attrs["n_p"], out.attrs["n_b"] = n_p, n_b
    return out

print("helpers loaded:", ", ".join(["fetch_gene", "numeric", "to_row", "clinvar_label",
                                    "calibrate", "strength_for", "worst_effect"]))

helpers loaded: fetch_gene, numeric, to_row, clinvar_label, calibrate, strength_for, worst_effect


In [3]:
MY_GENE = "CYP2C19"        # assigned/chosen gene 
assert MY_GENE in GENES, f"MY_GENE must be one of {GENES}"
MY_FILE = DATA / f"{MY_GENE}.tsv"
print("working on", MY_GENE)


working on CYP2C19


In [5]:
# Q1 - fetch, cache, label, and quality-check

if MY_FILE.exists() and not FORCE_REFETCH:
    G = pd.read_csv(MY_FILE, sep="\t")
    print(f"loaded cache: {MY_FILE}  ({len(G)} rows)")
else:
    G = fetch_gene(MY_GENE)
    G.to_csv(MY_FILE, sep="\t", index=False)
    print(f"fetched and cached to {MY_FILE}")

G = numeric(G)
G["label"] = G.clinvar_sig.map(clinvar_label)

# ---- check 1: how many variants came back? ----------------------------------
print(f"\n[1] COUNT: {len(G)} variants returned for {MY_GENE}")
if len(G) >= 1000:
    print("    [!] at or above the 1000-result cap: some CADD scores may be missing")
elif len(G) < 50:
    print("    [!] suspiciously few - check the gene symbol and re-fetch")
print("\n    ClinVar label breakdown:")
print(G.label.value_counts().to_string().replace("\n", "\n    "))

# ---- check 2: did the known star alleles arrive? ----------------------------
my_anchors = {rs: name for rs, name in ANCHORS.items() if name.startswith(MY_GENE)}
print(f"\n[2] ANCHORS expected for {MY_GENE}: {len(my_anchors)}")
for rs, name in my_anchors.items():
    hit = G[G.rsid == rs]
    if len(hit):
        r = hit.iloc[0]
        print(f"    FOUND   {rs:12s} {name:18s} label={r.label:12s} "
              f"CADD={r.cadd_phred}  gnomAD_AF={r.gnomad_af}")
    else:
        print(f"    MISSING {rs:12s} {name:18s}  <- fetch is incomplete, delete the cache and retry")

# ---- check 3: are the score columns actually populated? ---------------------
print("\n[3] SCORE COLUMN COVERAGE")
for c in ["cadd_phred", "revel", "alphamissense", "gnomad_af"]:
    n = G[c].notna().sum()
    print(f"    {c:15s} {n:5d}/{len(G)} ({100*n/len(G):5.1f}%)")


loaded cache: data\CYP2C19.tsv  (80 rows)

[1] COUNT: 80 variants returned for CYP2C19

    ClinVar label breakdown:
label
    other    38
    VUS      33
    B/LB      9

[2] ANCHORS expected for CYP2C19: 2
    FOUND   rs4244285    CYP2C19*2          label=other        CADD=5.686  gnomAD_AF=0.167953
    FOUND   rs12248560   CYP2C19*17         label=B/LB         CADD=3.031  gnomAD_AF=0.20012

[3] SCORE COLUMN COVERAGE
    cadd_phred         68/80 ( 85.0%)
    revel              54/80 ( 67.5%)
    alphamissense      54/80 ( 67.5%)
    gnomad_af          55/80 ( 68.8%)


In [6]:
# Q2 - tool coverage, broken down by consequence class

TOOLS = ["cadd_phred", "revel", "alphamissense"]
if "avi" in G.columns and G.avi.notna().any():
    TOOLS = TOOLS + ["avi"]

g = G.groupby(G.consequence.fillna("(none reported)"), dropna=False)
cov = g[TOOLS].apply(lambda d: d.notna().sum())
cov.insert(0, "n_variants", g.size())
for t in TOOLS:
    cov[t + "_%"] = (100 * cov[t] / cov.n_variants).round(1)

print(f"tool coverage by consequence class in {MY_GENE}\n")
print(cov.sort_values("n_variants", ascending=False).to_string())

print("\noverall percentage of variants each tool could score")
for t in TOOLS:
    n = G[t].notna().sum()
    print(f"  {t:15s} {n:5d}/{len(G)} = {100*n/len(G):5.1f}%")

# which classes no missense predictor can touch at all
print("\nconsequence classes with ZERO REVEL scores (REVEL is missense-only):")
zero_revel = cov[cov.revel == 0]
for cls, row in zero_revel.sort_values("n_variants", ascending=False).iterrows():
    print(f"  {cls:30s} {int(row.n_variants):4d} variants   "
          f"CADD scored {int(row.cadd_phred):4d}")

print("\nclasses where NO missense tool (REVEL or AlphaMissense) scored anything:")
none_missense = cov[(cov.revel == 0) & (cov.alphamissense == 0)]
print("  " + ", ".join(f"{c} (n={int(none_missense.loc[c,'n_variants'])})"
                       for c in none_missense.index) if len(none_missense) else "  (none)")
print("\nA blank cell means the tool COULD NOT SCORE the variant.")
print("It is not a benign result.")


tool coverage by consequence class in CYP2C19

                       n_variants  cadd_phred  revel  alphamissense  cadd_phred_%  revel_%  alphamissense_%
consequence                                                                                                
missense_variant               54          53     53             54          98.1     98.1            100.0
(none reported)                 8           0      0              0           0.0      0.0              0.0
synonymous_variant              7           6      0              0          85.7      0.0              0.0
upstream_gene_variant           5           4      0              0          80.0      0.0              0.0
intergenic_region               1           1      0              0         100.0      0.0              0.0
intron_variant                  1           1      0              0         100.0      0.0              0.0
start_lost                      1           1      1              0         100.0    100.

In [7]:
# Q3 - frequency filter with a threshold defensible for a PHARMACOGENE

# CYP2C19 reduced- and increased-function star alleles are common, not rare:
# *2 (rs4244285) sits near 15% globally and ~30% in East Asian populations,
# *17 (rs12248560) near 20% in Europeans. A rare-disease ceiling would delete
# exactly the variants that matter. The threshold is set above the highest
# known functional allele frequency in this gene, and is used only to drop
# variants too common to explain an individual drug-response phenotype.
MAX_AF = 0.25

n_before = len(G)
too_common = G[G.gnomad_af > MAX_AF]
print(f"MAX_AF = {MAX_AF}")
print(f"removes {len(too_common)} of {n_before} variants "
      f"({100*len(too_common)/n_before:.1f}%)")
print(f"  (variants with no gnomAD AF at all are KEPT: {G.gnomad_af.isna().sum()} rows)")

# (c) CRITICAL - what clinically important variants does my filter throw away?
lost = too_common[too_common.label.isin(["P/LP", "other"])]
print(f"\nP/LP or drug-response variants removed by MAX_AF={MAX_AF}: {len(lost)}")
cols = ["rsid", "consequence", "aa_change", "gnomad_af", "clinvar_sig", "label"]
print(lost[cols].to_string(index=False) if len(lost) else "  (none - the threshold keeps every labelled functional allele)")

# (d) contrast: what a rare-disease filter would have destroyed
RARE = 0.001
lost_rare = G[(G.gnomad_af > RARE) & G.label.isin(["P/LP", "other"])]
print(f"\nFor contrast, a {RARE} rare-disease filter would remove "
      f"{(G.gnomad_af > RARE).sum()} variants, including "
      f"{len(lost_rare)} P/LP or drug-response variants:")
print(lost_rare[cols].to_string(index=False) if len(lost_rare) else "  (none)")

# do the anchors survive my threshold?
print(f"\nanchor alleles under MAX_AF = {MAX_AF}:")
for rs, name in my_anchors.items():
    h = G[G.rsid == rs]
    if len(h):
        af = h.iloc[0].gnomad_af
        kept = not (af is not None and af == af and af > MAX_AF)
        print(f"  {name:18s} AF={af}  ->  {'KEPT' if kept else 'REMOVED'}")

# (e) apply it
G = G[~(G.gnomad_af > MAX_AF)].copy()
print(f"\nG is now {len(G)} variants after filtering")


MAX_AF = 0.25
removes 2 of 80 variants (2.5%)
  (variants with no gnomAD AF at all are KEPT: 25 rows)

P/LP or drug-response variants removed by MAX_AF=0.25: 0
  (none - the threshold keeps every labelled functional allele)

For contrast, a 0.001 rare-disease filter would remove 19 variants, including 14 P/LP or drug-response variants:
       rsid           consequence aa_change  gnomad_af   clinvar_sig label
  rs7902257 upstream_gene_variant       NaN   0.022133 drug response other
 rs41291556      missense_variant     W120R   0.001571         other other
 rs28399504            start_lost       M1V   0.002379 drug response other
  rs4986893           stop_gained     W212X   0.002506 drug response other
  rs4244285    synonymous_variant       NaN   0.167953 drug response other
 rs17882687      missense_variant      I19L   0.005733 drug response other
 rs17879685      missense_variant     R410C   0.004596 drug response other
 rs17884712      missense_variant     R144H   0.003570 drug re

In [8]:
# Q4 - calibrate a predictor into ACMG evidence strength

# CADD is chosen over REVEL because REVEL is missense-only: the Q2 audit shows it
# scored 0/7 synonymous, 0/5 upstream, 0/1 splice-donor and 0/1 intronic variants
# in this gene. CADD scored 85% of all variants, which is what makes a gene-wide
# calibration possible at all.
SCORE_COL = "cadd_phred"
BANDS     = [0, 10, 20, 25, 30, 50]

# ---- attempt 1: CYP2C19 on its own -----------------------------------------
cal_own = calibrate(G, SCORE_COL, BANDS)
n_p_own, n_b_own = cal_own.attrs["n_p"], cal_own.attrs["n_b"]
print(f"ATTEMPT 1 - calibrating {SCORE_COL} on {MY_GENE} alone\n")
print(cal_own.to_string())
print(f"\nrests on {n_p_own + n_b_own} labelled variants: "
      f"{n_p_own} P/LP and {n_b_own} B/LB")

CALIBRATION_SOURCE = MY_GENE
if n_p_own < 10 or n_b_own < 10:
    print(f"\n[!] NOT CALIBRATABLE ALONE. {MY_GENE} has {n_p_own} P/LP and {n_b_own} B/LB")
    print("    variants after filtering. An LR+ built on this is a zero numerator or a")
    print("    5-variant denominator, not evidence. Most CYP2C19 ClinVar entries are")
    print("    submitted as 'drug response', which is not a P/LP assertion, so the")
    print("    pathogenic side of the calibration is empty by construction.")
    print("\n    Taking the documented fallback: POOL with the other four pharmacogenes.")

    # ---- attempt 2: pool across the pharmacogene panel ----------------------
    frames = [G.assign(source_gene=MY_GENE)]
    for gn in [x for x in GENES if x != MY_GENE]:
        f = DATA / f"{gn}.tsv"
        if f.exists() and not FORCE_REFETCH:
            d = pd.read_csv(f, sep="\t")
            print(f"  {gn:9s} loaded from cache ({len(d)} rows)")
        else:
            d = fetch_gene(gn)
            d.to_csv(f, sep="\t", index=False)
        d = numeric(d)
        d["label"] = d.clinvar_sig.map(clinvar_label)
        d = d[~(d.gnomad_af > MAX_AF)]        # same frequency ceiling as Q3
        frames.append(d.assign(source_gene=gn))

    POOL = pd.concat(frames, ignore_index=True).drop_duplicates("hgvs_g")
    print(f"\npooled set: {len(POOL)} variants across {POOL.source_gene.nunique()} genes")
    lab = POOL[POOL.label.isin(["P/LP", "B/LB"]) & POOL[SCORE_COL].notna()]
    print("\nlabelled variants contributed by each gene (the calibration set)\n")
    print(pd.crosstab(lab.source_gene, lab.label).to_string())

    cal = calibrate(POOL, SCORE_COL, BANDS)
    CALIBRATION_SOURCE = "pooled pharmacogene panel"
else:
    cal = cal_own
    POOL = G

n_p, n_b = cal.attrs["n_p"], cal.attrs["n_b"]
print(f"\n{'='*70}")
print(f"CALIBRATION IN USE: {SCORE_COL} on the {CALIBRATION_SOURCE}\n")
print(cal.to_string())
print(f"\nrests on {n_p + n_b} labelled variants: {n_p} P/LP and {n_b} B/LB")

# (c) which bands are too thin to believe?
thin = cal[cal.n < 10]
print(f"\nbands resting on fewer than 10 labelled variants: {len(thin)} of {len(cal)}")
if len(thin):
    print(thin.to_string())
    print("\nThe LR+ in these bands is not evidence. Quote them with the denominator")
    print("attached, or decline to use them.")

inf_bands = cal[np.isinf(cal["LR+"].astype(float))]
if len(inf_bands):
    print(f"\n{len(inf_bands)} band(s) have LR+ = inf because n_BLB = 0.")
    print("That is a zero denominator, not infinite evidence.")

print("\nCOST OF THE FALLBACK: these LRs describe how CADD separates pathogenic from")
print("benign variants across five pharmacogenes, not within CYP2C19. Applying them")
print("to CYP2C19 assumes CADD behaves the same way in this gene as in DPYD and")
print("TPMT, which is an assumption this analysis cannot test.")


ATTEMPT 1 - calibrating cadd_phred on CYP2C19 alone

          n  n_PLP  n_BLB  LR+        evidence
band                                          
(0, 10]   5      0      5  0.0  strong, benign
(10, 20]  0      0      0  NaN       undefined
(20, 25]  0      0      0  NaN       undefined
(25, 30]  0      0      0  NaN       undefined
(30, 50]  0      0      0  NaN       undefined

rests on 5 labelled variants: 0 P/LP and 5 B/LB

[!] NOT CALIBRATABLE ALONE. CYP2C19 has 0 P/LP and 5 B/LB
    variants after filtering. An LR+ built on this is a zero numerator or a
    5-variant denominator, not evidence. Most CYP2C19 ClinVar entries are
    submitted as 'drug response', which is not a P/LP assertion, so the
    pathogenic side of the calibration is empty by construction.

    Taking the documented fallback: POOL with the other four pharmacogenes.
  CYP2C9       41 variants found  ->  41 annotated
  VKORC1       53 variants found  ->  53 annotated
  DPYD        426 variants found  ->  426 an

In [11]:
# Q5 - apply the calibrated evidence to the VUS rows and export

vus = G[G.label == "VUS"].copy()
print(f"{len(vus)} VUS in {MY_GENE} after filtering "
      f"({vus[SCORE_COL].notna().sum()} of them have a {SCORE_COL} score)")
print(f"evidence bands come from: {CALIBRATION_SOURCE}\n")

vus["band"]      = pd.cut(vus[SCORE_COL], BANDS)
vus["lr"]        = pd.to_numeric(vus.band.map(cal["LR+"].to_dict()).astype(object), errors="coerce")
vus["evidence"]  = vus.band.map(cal["evidence"].to_dict()).astype(object)
vus["n_in_band"] = pd.to_numeric(vus.band.map(cal["n"].to_dict()).astype(object), errors="coerce")

# a band built on fewer than 10 labelled variants cannot move anything
THIN = 10
vus.loc[vus.n_in_band < THIN, "evidence"] = "band too small to use"
vus.loc[vus[SCORE_COL].isna(), "evidence"] = "not scored"

def provisional(e):
    if not isinstance(e, str):                       return "remains VUS"
    if e == "not scored":                            return "remains VUS (no score)"
    if e == "band too small to use":                 return "remains VUS (band too small)"
    if e in ("no evidence either way", "undefined"): return "remains VUS"
    if "pathogenic" in e:                            return "towards pathogenic"
    if "benign" in e:                                return "towards benign"
    return "remains VUS"

vus["provisional_call"] = vus.evidence.map(provisional)

print("where the VUS moved\n")
print(vus.provisional_call.value_counts().to_string())
print("\nA VUS whose score lands in a band with LR+ near 1 STAYS a VUS: the")
print("predictor genuinely told us nothing. A VUS in a band built on fewer than")
print(f"{THIN} labelled variants also stays put, because that LR+ is noise.")

print("\nany VUS that moved, with the band size behind the call\n")
moved = vus[vus.provisional_call.str.startswith("towards")]
mcols = ["rsid", "consequence", "aa_change", SCORE_COL, "gnomad_af",
         "lr", "n_in_band", "evidence", "provisional_call"]
print(moved[mcols].sort_values(SCORE_COL, ascending=False).head(20).to_string(index=False)
      if len(moved) else "  (none moved)")

print("\nthe VUS nothing could score at all (not a benign result)\n")
uns = vus[vus[SCORE_COL].isna()]
print(uns[["rsid", "consequence", "aa_change", "gnomad_af"]].to_string(index=False)
      if len(uns) else "  (none)")

OUT = f"pgx_classified_{MY_GENE}.csv"
vus[["rsid", "gene", "consequence", "aa_change", SCORE_COL, "gnomad_af",
     "evidence", "provisional_call"]].to_csv(OUT, index=False)
print(f"\nwrote {OUT}  ({len(vus)} rows)")


33 VUS in CYP2C19 after filtering (27 of them have a cadd_phred score)
evidence bands come from: pooled pharmacogene panel

where the VUS moved

provisional_call
towards benign                  16
towards pathogenic               8
remains VUS (no score)           6
remains VUS (band too small)     3

A VUS whose score lands in a band with LR+ near 1 STAYS a VUS: the
predictor genuinely told us nothing. A VUS in a band built on fewer than
10 labelled variants also stays put, because that LR+ is noise.

any VUS that moved, with the band size behind the call

        rsid      consequence aa_change  cadd_phred  gnomad_af   lr  n_in_band               evidence   provisional_call
         NaN missense_variant     D414H      24.100        NaN 2.17       17.0 supporting, pathogenic towards pathogenic
 rs371309036 missense_variant     L131P      24.000   0.000007 2.17       17.0 supporting, pathogenic towards pathogenic
rs1848387818 missense_variant      Y61C      23.900        NaN 2.17      

In [12]:
SKIP_ATLAS   = True
MAX_VARIANTS = 200

import os
ATLAS_STATUS = "not run"

def to_variant_string(hgvs):
    """Atlas wants 1-based GRCh38 chr:pos:ref>alt. It does NOT accept rsIDs."""
    m = re.match(r"(chr[\w]+):g\.(\d+)([ACGT])>([ACGT])", str(hgvs))
    return f"{m.group(1)}:{m.group(2)}:{m.group(3)}>{m.group(4)}" if m else None

if SKIP_ATLAS:
    print("SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.")
    ATLAS_STATUS = "skipped"
else:
    try:
        from alphagenome.atlas import atlas
        from alphagenome.data import genome
        key = os.environ.get("ALPHAGENOME_API_KEY") or os.environ.get("ALPHA_GENOME_API_KEY")
        if not key:
            raise RuntimeError(
                "No API key in your environment. Did you relaunch Jupyter from a NEW\n"
                "  terminal after setting it? Check with:\n"
                "    import os; print(os.environ.get('ALPHAGENOME_API_KEY'))\n"
                "  Or set SKIP_ATLAS = True.")
        client = atlas.create(key, timeout=30)

        pri = G.assign(_nc=~G.consequence.fillna("").str.contains("missense")) \
               .sort_values("_nc", ascending=False).head(MAX_VARIANTS)
        rows, errs = [], 0
        for n, (_, r) in enumerate(pri.iterrows(), 1):
            vs = to_variant_string(r.hgvs_g)
            if not vs:
                continue
            try:
                res = client.query_variant(
                    genome.Variant.from_str(vs),
                    requested_scorers=["AVI_SCORE", "AVI_SCORE_FEATURE_IMPORTANCE"])
                ad = res["AVI_SCORE"]
                cdf = float(ad.layers["quantiles"][0, 0])
                fi = res["AVI_SCORE_FEATURE_IMPORTANCE"]
                rows.append({"rsid": r.rsid,
                             "avi_phred": round(-10*np.log10(max(1-cdf, 1e-12)), 2),
                             "top_feature": fi.var["name"].values[int(np.argmax(np.abs(fi.X[0])))]})
            except Exception:
                errs += 1
            if n % 50 == 0:
                print(f"  {n}/{len(pri)} queried", flush=True)

        avi = pd.DataFrame(rows)
        if len(avi):
            avi.to_csv(DATA / f"avi_{MY_GENE}.tsv", sep="\t", index=False)
            G = G.merge(avi, on="rsid", how="left")
            G["avi"] = G.avi_phred
            ATLAS_STATUS = "queried live"
            print(f"\nscored {len(avi)} variants ({errs} failures)")
    except ImportError:
        print('alphagenome not installed. pip install "alphagenome>=0.9.0", then restart.')
        print("Or set SKIP_ATLAS = True.")
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

if "avi" not in G.columns: G["avi"] = np.nan
if "top_feature" not in G.columns: G["top_feature"] = np.nan
G["avi"] = pd.to_numeric(G["avi"], errors="coerce")

if G.avi.notna().sum() > 0:
    a = G[G.avi.notna()]
    cols = [c for c in ["rsid", "consequence", "avi_phred", "top_feature",
                        "revel", "cadd_phred", "label"] if c in a.columns]
    print(f"\nhighest-impact {MY_GENE} variants by AVI\n")
    print(a.sort_values("avi", ascending=False)[cols].head(10).to_string(index=False))
    print("\nwhat is driving the scores (read this BEFORE the numbers)")
    print(a.top_feature.value_counts().head(8).to_string())
else:
    nc = G[~G.consequence.fillna("").str.contains("missense")]
    print(f"\nnon-missense variants in {MY_GENE}, which is what the Atlas is for: {len(nc)}")
    print(nc.consequence.value_counts().to_string())

print(f"\n[Atlas status: {ATLAS_STATUS}]")

SKIPPED. Disclose this in your report and answer Q6.2 from reasoning.

non-missense variants in CYP2C19, which is what the Atlas is for: 25
consequence
synonymous_variant       6
upstream_gene_variant    5
start_lost               1
stop_gained              1
intron_variant           1
splice_donor_variant     1
stop_lost                1
intergenic_region        1

[Atlas status: skipped]


In [13]:
d = G.copy()
coding = d.consequence.isin(["missense_variant", "synonymous_variant"])

d["flag_cadd_vs_revel"]  = (d.cadd_phred > 25) & (d.revel < 0.35)
d["flag_revel_vs_am"]    = (d.revel - d.alphamissense).abs() > 0.4
d["flag_scored_but_vus"] = (d.label == "VUS") & (d.revel > 0.7)
d["flag_common_but_plp"] = (d.gnomad_af > 0.01) & (d.label.isin(["P/LP", "other"]))
d["flag_no_tool_at_all"] = d[["revel", "alphamissense", "cadd_phred"]].isna().all(axis=1)
if d.avi.notna().any():
    d["flag_atlas_only"] = (d.avi > 20) & d.revel.isna() & d.alphamissense.isna()

flags = [c for c in d.columns if c.startswith("flag_")]
print(f"discordant variants in {MY_GENE}\n")
print(d[flags].sum().to_string())

hits = d[d[flags].any(axis=1)]
cols = [c for c in ["rsid", "note", "consequence", "consequence_all", "aa_change",
                    "cadd_phred", "revel", "alphamissense", "avi", "gnomad_af",
                    "label"] if c in d.columns]
print(f"\n{len(hits)} variants trip at least one rule\n")
print(hits[cols].head(15).to_string(index=False) if len(hits) else "  (none)")

if len(hits) == 0:
    print("\nNone. That is a legitimate result. Say so in your report and explain why")
    print("this gene's variants are concordant, rather than loosening the rules until")
    print("something fires.")

discordant variants in CYP2C19

flag_cadd_vs_revel      3
flag_revel_vs_am        6
flag_scored_but_vus     0
flag_common_but_plp     4
flag_no_tool_at_all    10

21 variants trip at least one rule

       rsid           consequence                         consequence_all aa_change  cadd_phred  revel  alphamissense  avi  gnomad_af label
        NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN  NaN        NaN   VUS
  rs7902257 upstream_gene_variant intergenic_region;upstream_gene_variant       NaN         NaN    NaN            NaN  NaN   0.022133 other
        NaN                   NaN                                     NaN       NaN         NaN    NaN            NaN  NaN        NaN  B/LB
 rs41291556      missense_variant                        missense_variant     W120R      25.700  0.440         0.9402  NaN   0.001571 other
rs145328984      missense_variant                        missense_variant      R73C      26.000  0.12